#AIRFLOW Incremental Load DAG

In [0]:
from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime
import requests
import json
import os
from requests.auth import HTTPBasicAuth
from requests.exceptions import RequestException, Timeout, HTTPError


def fetch_api_data(start_date, end_date, output_dir, output_file):
    """
    Fetch data from API for a date range and save to file.
    Dates and filename are injected by Airflow via Jinja templating.
    """

    url = 'http://fastapi-app:5000/getAll'

    headers = {
        'accept': 'application/json',
        'Content-Type': 'application/json'
    }

    payload = {
        "start_date": start_date,
        "end_date": end_date,
        "limit": 10
    }

    print(f"Fetching data for date range: {start_date} to {end_date}")

    response = None
    try:
        response = requests.post(
            url,
            headers=headers,
            json=payload,
            auth=HTTPBasicAuth('admin', 'manish'),
            timeout=30
        )

        response.raise_for_status()
        data = response.json()
        print(f"Success! Got {len(data)} records")

        # Ensure the directory exists
        os.makedirs(output_dir, exist_ok=True)

        # Build full path from templated directory + filename
        full_path = os.path.join(output_dir, output_file)

        with open(full_path, "w") as file:
            json.dump(data, file, indent=4)

        print(f"Data saved to: {full_path}")
        return full_path  # Return path so downstream tasks can use it

    except HTTPError as e:
        print(f"HTTP Error: {e}")
        if response is not None:
            print(f"Response body: {response.text}")
        raise
    except Timeout:
        print("Request timed out after 30 seconds")
        raise
    except RequestException as e:
        print(f"Request failed: {e}")
        raise
    except ValueError:
        print("Response wasn't valid JSON")
        if response is not None:
            print(response.text)
        raise


with DAG(
        dag_id="incremental_load_dag",
        start_date=datetime(2026, 5, 1),
        schedule='0 0 * * *',
        catchup=True, #Concept of Backfill
        tags=['api', 'incremental']
) as dag:
    pull_data_python_task = PythonOperator(
        task_id="pull_data_python_task",
        python_callable=fetch_api_data,
        op_kwargs={
            'start_date': '{{ ds }}',                       # e.g. 2026-05-13
            'end_date': '{{ ds }}',                         # e.g. 2026-05-13
            'output_dir': '/opt/airflow/output_files',
            'output_file': 'data_{{ ts_nodash }}.json'      # e.g. data_20260513T000000.json
        }
    )

#CODE EXPLANATION

In [0]:
from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime
import requests
import json
import os
from requests.auth import HTTPBasicAuth
from requests.exceptions import RequestException, Timeout, HTTPError

###from airflow.sdk import DAG
Imports the DAG class, which is the container that defines your workflow. Used in with DAG(dag_id="incremental_load_dag", ...) as dag: — every Airflow workflow must be wrapped in a DAG object so the scheduler knows it exists, when to run it, and what tasks belong to it.

###from airflow.providers.standard.operators.python import PythonOperator
Imports the operator that lets you run a regular Python function as an Airflow task. Used in pull_data_python_task = PythonOperator(...) to wrap your fetch_api_data function so Airflow can schedule, retry, log, and template-render it. Without this, Airflow has no way to execute arbitrary Python code as a task.

##Standard library imports

###from datetime import datetime
Provides the datetime class for specifying when the DAG should start. Used in start_date=datetime(2026, 5, 1) — Airflow needs a concrete datetime object (not a string) to anchor the schedule and compute interval boundaries like ds and ts_nodash.

###import requests
The HTTP client library used to call your FastAPI service. Used in requests.post(url, headers=headers, json=payload, ...) inside fetch_api_data. This is what actually makes the network call to http://fastapi-app:5000/getAll.

###import json
Used to serialize the API response (a Python dict/list) into a JSON-formatted file. Used in json.dump(data, file, indent=4) when writing the output file. The indent=4 makes the saved file human-readable. Note: you don't need json for sending the payload because requests handles that via the json=payload argument.

response = requests.post(
    url,
    headers=headers,
    json=payload,   # ← this is just a keyword argument name, NOT the json module
    ...
)

Here, json=payload is just a parameter name in the requests.post() function signature. It doesn't reference the json module at all. The requests library internally serializes payload to JSON for you.

with open(full_path, "w") as file:
    json.dump(data, file, indent=4)   # ← this DOES use the json module

Here, json.dump(...) is calling the dump function from the json module. This line will break with a NameError if you remove import json.


###import os
Provides filesystem utilities. Used in two places:

os.makedirs(output_dir, exist_ok=True) — creates the output directory if it doesn't exist, without erroring if it does.
os.path.join(output_dir, output_file) — safely combines the directory and filename into a full path using the correct separator for the OS.

##Auth and exception imports from requests
###from requests.auth import HTTPBasicAuth
Provides the helper class for HTTP Basic Authentication. Used in auth=HTTPBasicAuth('admin', 'gaurav') in your POST call. Your FastAPI endpoint requires basic auth credentials, and this class formats the username/password into the proper Authorization header automatically.
###from requests.exceptions import RequestException, Timeout, HTTPError
Imports the three specific exception types you handle in your try/except blocks. They form a hierarchy that lets you catch errors at different levels of specificity:

HTTPError — raised by response.raise_for_status() when the API returns a 4xx or 5xx status (e.g., 401 unauthorized, 500 server error). Caught first so you can log the response body for debugging.

Timeout — raised when the request exceeds your 30-second timeout. Caught separately so you can log a specific message about the timeout.

RequestException — the parent class for all requests errors (connection refused, DNS failure, SSL issues, etc.). Caught last as a catch-all for any network problem that isn't an HTTPError or Timeout.

Order matters here: Python checks except blocks top-to-bottom, so the most specific exceptions (HTTPError, Timeout) must come before the general RequestException, otherwise the general one would catch everything first.

In [0]:
def fetch_api_data(start_date, end_date, output_dir, output_file):
    """
    Fetch data from API for a date range and save to file.
    Dates and filename are injected by Airflow via Jinja templating.
    """

    url = 'http://fastapi-app:5000/getAll'

    headers = {
        'accept': 'application/json',
        'Content-Type': 'application/json'
    }

    payload = {
        "start_date": start_date,
        "end_date": end_date,
        "limit": 10
    }

    print(f"Fetching data for date range: {start_date} to {end_date}")

    response = None

###def fetch_api_data(start_date, end_date, output_dir, output_file):
Defines the function that will be executed as your Airflow task. The four parameters match exactly the four keys you pass in op_kwargs in the DAG:

op_kwargs={
    'start_date': '{{ ds }}',
    'end_date': '{{ ds }}',
    'output_dir': '/opt/airflow/output_files',
    'output_file': 'data_{{ ts_nodash }}.json'
}
By the time this function actually runs, Airflow has already rendered the Jinja templates, so inside the function start_date is a plain string like "2026-05-13", not "{{ ds }}". The function itself knows nothing about Jinja or Airflow — it just receives strings.

###url = 'http://fastapi-app:5000/getAll'
The target endpoint. The hostname fastapi-app is a Docker Compose service name — it only resolves inside the Docker network where both your Airflow containers and the FastAPI container live. If you ran this script outside Docker, it would fail with a DNS error. Port 5000 and the /getAll path are specific to your FastAPI app.

###Request Headers
headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json'
}

Two HTTP headers attached to the request:

accept: application/json tells the server "I want the response in JSON format." If your API supports multiple formats (XML, CSV, etc.), this picks JSON.
Content-Type: application/json tells the server "the body I'm sending you is JSON." Without it, some servers will refuse to parse the body or will misinterpret it as form data.

Note: requests would actually set Content-Type automatically when you use the json= parameter, so this header is somewhat redundant — but being explicit doesn't hurt and makes the intent obvious.

###Request Payload
payload = {
    "start_date": start_date,
    "end_date": end_date,
    "limit": 10
}

The data you'll send to the API in the POST body. It tells the API: "give me records between these two dates, up to 10 of them." The start_date and end_date values here are the templated dates passed into the function. limit: 10 is hardcoded — if you wanted to fetch more records per day you'd raise this number or parameterize it too.

###Note
n Airflow, anything you print() from inside a task gets captured into the task's log, viewable from the UI. This is your breadcrumb for debugging — when you look at a backfilled run later, this line tells you which date range that specific run was processing. Very useful when 12 backfill runs are executing and you need to figure out what each one did.

In [0]:
    try:
        response = requests.post(
            url,
            headers=headers,
            json=payload,
            auth=HTTPBasicAuth('admin', 'manish'),
            timeout=30
        )

        response.raise_for_status()
        data = response.json()
        print(f"Success! Got {len(data)} records")

        # Ensure the directory exists
        os.makedirs(output_dir, exist_ok=True)

        # Build full path from templated directory + filename
        full_path = os.path.join(output_dir, output_file)

        with open(full_path, "w") as file:
            json.dump(data, file, indent=4)

        print(f"Data saved to: {full_path}")
        return full_path  # Return path so downstream tasks can use it

    except HTTPError as e:
        print(f"HTTP Error: {e}")
        if response is not None:
            print(f"Response body: {response.text}")
        raise
    except Timeout:
        print("Request timed out after 30 seconds")
        raise
    except RequestException as e:
        print(f"Request failed: {e}")
        raise
    except ValueError:
        print("Response wasn't valid JSON")
        if response is not None:
            print(response.text)
        raise


###Making the HTTP request
response = requests.post(
    url,
    headers=headers,
    json=payload,
    auth=HTTPBasicAuth('admin', 'gaurav'),
    timeout=30
)

Sends a POST request to your FastAPI endpoint. Five things are happening in this one call:
url — where the request goes (http://fastapi-app:5000/getAll).

headers=headers — attaches the accept and Content-Type headers you defined.

json=payload — tells requests to serialize your Python dict to JSON and put it in the request body. This is the parameter name we discussed earlier — it's not the json module.

auth=HTTPBasicAuth('admin', 'gaurav') — adds an Authorization: Basic <base64-encoded-credentials> header. Your FastAPI endpoint requires basic auth, and this is how requests handles it.

timeout=30 — if the server doesn't respond within 30 seconds, raise a Timeout exception. Without this, a hung server could make your Airflow task hang forever, occupying a worker slot indefinitely. This is one of the most important arguments in production code.

The returned Response object contains the status code, headers, and body — but at this point we don't yet know if the call succeeded.

###Checking for errors
response.raise_for_status()

By default, requests does not treat a 4xx or 5xx response as an error — it just returns the response normally with response.status_code == 404 (or whatever). That's surprising behavior and a common source of bugs.

raise_for_status() fixes this by inspecting the status code and raising HTTPError if it's 4xx or 5xx. So this single line converts "silently got a 401" into "loud, catchable exception." If you skip it, your code would happily try to parse an error page as data and produce confusing downstream failures.

###Parsing Response
data = response.json()

Deserializes the response body from JSON text into Python objects (dict or list). If the body isn't valid JSON — say the server returned HTML or plain text — this raises ValueError (specifically json.JSONDecodeError, which is a subclass of ValueError). That's why there's a except ValueError block later.

###os.makedirs(output_dir, exist_ok=True)

Creates /opt/airflow/output_files if it doesn't exist. The exist_ok=True flag is critical — without it, this call would raise FileExistsError on the second and every subsequent run, because the directory would already be there. With the flag, it's a no-op if the directory exists, and creates it if it doesn't. Idempotent.

###full_path = os.path.join(output_dir, output_file)
Combines /opt/airflow/output_files with data_20260513T000000.json into /opt/airflow/output_files/data_20260513T000000.json. Using os.path.join instead of string concatenation (output_dir + "/" + output_file) is more portable and correctly handles edge cases like trailing slashes.

with open(...) as file: — opens the file in write mode ("w", which truncates any existing file with the same name). The with statement ensures the file is properly closed even if an exception occurs during writing. Always prefer this over manual open()/close().
json.dump(data, file, indent=4) — writes the Python object to the file as JSON, pretty-printed with 4-space indentation. indent=4 is purely for readability; without it, the entire JSON would be one long line.

In Airflow, anything a PythonOperator's callable returns is automatically pushed to XCom (cross-communication) under the key return_value. This means downstream tasks in the same DAG run can pull this path with {{ ti.xcom_pull(task_ids='pull_data_python_task') }} and know exactly where the file lives — without hardcoding the path or recomputing it. Even though you don't have downstream tasks yet, this sets you up for them.

#4xx or 5xx
except HTTPError as e:
    print(f"HTTP Error: {e}")
    if response is not None:
        print(f"Response body: {response.text}")
    raise

Triggered by raise_for_status() when the server explicitly rejected your request (401 unauthorized, 404 not found, 500 server error, etc.). In this case:

You got a response back, so response is not None.
The response body usually contains an error message from the API ("invalid credentials", "date out of range", etc.) — response.text prints that raw body so you can see exactly what the server complained about.
raise re-raises the exception so Airflow knows the task failed. Without raise, the exception would be silently swallowed and Airflow would mark the task as successful, which is dangerous — you'd have a broken pipeline that looks healthy.

except Timeout:
    print("Request timed out after 30 seconds")
    raise

Triggered when 30 seconds pass without a response. No response object exists (the request never completed), so there's nothing to log from response.text. Just log the cause and re-raise.
This being a separate handler — instead of being lumped into RequestException — lets you give a clear, specific log message and makes it easy to add timeout-specific logic later (e.g., longer retry delays for timeouts vs. immediate retries for other network errors).

##Network Problem
except RequestException as e:
    print(f"Request failed: {e}")
    raise

The catch-all for the requests library. RequestException is the parent class of all requests-specific errors, including:

ConnectionError — DNS failure, refused connection, etc.
TooManyRedirects
SSLError
Any other network-layer failure not caught above

Since HTTPError and Timeout are subclasses of RequestException, the order matters: if this block were first, it would catch HTTPErrors and Timeouts too, and your specific handlers above would never run.

###Invalid Json

except ValueError:
    print("Response wasn't valid JSON")
    if response is not None:
        print(response.text)
    raise

Triggered by response.json() when the body can't be parsed. Common causes: the server returned HTML (often an error page from a proxy or load balancer), plain text, or empty content. Logging response.text shows you exactly what came back instead of JSON — critical for debugging because the HTTP status might even be 200 OK while the body is garbage.



In [0]:
with DAG(
        dag_id="incremental_load_dag",
        start_date=datetime(2026, 5, 1),
        schedule='0 0 * * *',
        catchup=True, #Concept of Backfill
        tags=['api', 'incremental']
) as dag:

The with statement creates a DAG object and makes it the "active" DAG for everything inside the block. Any operator you instantiate inside (like PythonOperator) is automatically attached to this DAG — you don't have to manually write dag=dag on every task.
If you didn't use with, you'd have to write:

dag = DAG(dag_id="incremental_load_dag", ...)
pull_data_python_task = PythonOperator(
    task_id="pull_data_python_task",
    ...,
    dag=dag,   # ← have to pass this explicitly every time
)

##DAG ARGUEMENTS

###dag_id="incremental_load_dag"
The unique identifier for this DAG across your entire Airflow instance. This is what appears in the UI, what you reference in CLI commands (airflow dags trigger incremental_load_dag), and what Airflow uses internally to track runs in the metadata database.

Must be unique — if two files define DAGs with the same dag_id, Airflow will log warnings and behave unpredictably. Naming convention: lowercase with underscores, descriptive of what the DAG does.

###start_date=datetime(2026, 5, 1)
The earliest date Airflow will consider for scheduling runs. Combined with the schedule, this defines the timeline of when runs could exist.
Critical detail: with catchup=True, Airflow will create one run for every scheduled interval between start_date and "now." So setting start_date=datetime(2026, 5, 1) and unpausing on May 13 → 12 backfilled runs will be queued.

Important gotcha: never change start_date after a DAG has been running. Airflow uses it to figure out which intervals already ran; moving it can cause runs to be re-created or skipped unexpectedly. If you want to re-run history, use the backfill CLI or clear runs in the UI instead.
schedule='0 0 * * *'

A cron expression telling Airflow how often to create runs. Reading the five fields left to right:
So this means: "every day at 00:00 UTC." A daily run at midnight.
You could also write this as schedule='@daily' — Airflow accepts preset shortcuts (@hourly, @daily, @weekly, @monthly, @yearly). They're equivalent but the cron form is more explicit and easier to tweak later (e.g., changing to 0 6 * * * for 6 AM).

###catchup=True, #Concept of Backfill
This is the switch that controls historical runs.

catchup=True → when you unpause the DAG, Airflow creates a run for every scheduled interval from start_date up to now. This is "backfill" — replaying history.
catchup=False → Airflow only schedules from the current moment forward; missed intervals are ignored.

For incremental data loads (like yours), catchup=True is usually what you want, because each run is responsible for one specific day of data — if you skip a day, you have a gap. The whole point of templating start_date and end_date with {{ ds }} is so that each run knows which historical date it's responsible for.
Your inline comment #Concept of Backfill is a reminder that this single flag is what enables that behavior.

###tags=['api', 'incremental']
Cosmetic labels that show up in the Airflow UI, letting you filter the DAG list. No runtime effect — purely for organization. As your Airflow instance grows to dozens of DAGs, tags make it easy to find "all the API-pulling DAGs" or "all the incremental loads."


In [0]:
    pull_data_python_task = PythonOperator(
        task_id="pull_data_python_task",
        python_callable=fetch_api_data,
        op_kwargs={
            'start_date': '{{ ds }}',                       # e.g. 2026-05-13
            'end_date': '{{ ds }}',                         # e.g. 2026-05-13
            'output_dir': '/opt/airflow/output_files',
            'output_file': 'data_{{ ts_nodash }}.json'      # e.g. data_20260513T000000.json
        }
    )

This is where you create one task inside the DAG. Every DAG is made of one or more tasks, and each task is an instance of some operator. PythonOperator is the operator that runs a Python function.

###task_id="pull_data_python_task"
The unique identifier for this task within this DAG. (Unlike dag_id, it only needs to be unique inside one DAG — you can have a pull_data_python_task in multiple DAGs without conflict.) Shows up as the node label in the graph view, in logs, and in CLI commands. Also used when other tasks pull this one's XCom values: xcom_pull(task_ids='pull_data_python_task').

###python_callable=fetch_api_data
Tells the operator which function to execute when the task runs. Note: you pass the function itself (fetch_api_data), not a call to it (fetch_api_data()). If you accidentally added the parentheses, Python would call the function immediately at DAG-parse time and pass the result to python_callable, which would break everything.

###op_kwargs={...}
A dictionary of keyword arguments to pass into fetch_api_data when the task runs. Each key matches a parameter name in the function signature:

###concept of op_kwargs and op_orgs


op_kwargs={
    'start_date': '{{ ds }}',
    'end_date': '{{ ds }}',
    'output_dir': '/opt/airflow/output_files',
    'output_file': 'data_{{ ts_nodash }}.json'
}

The names must match exactly, or you'll get a TypeError about unexpected/missing arguments when the task tries to run.
How Jinja templating connects to all of this

This is the real magic of the DAG block, and it's worth understanding clearly.
op_kwargs is listed in PythonOperator.template_fields. That means before Airflow invokes your function, it walks through every string value in op_kwargs and renders any {{ ... }} expressions using the current run's context.

So for the run with logical date 2026-05-13, this dictionary:

This is why your function knows nothing about Jinja or Airflow internals — it just receives plain strings. The templating is the bridge between Airflow's scheduler-level knowledge of "which date is this run for?" and your function's actual logic.

Each backfilled run gets a different value of ds, so each run produces a differently-named file with different date parameters in the payload — without you having to do anything special in the function.

###JINJA2
The complete list of Jinja variables Airflow makes available lives on one page in the official docs:https://airflow.apache.org/docs/apache-airflow/stable/templates-ref.html

This page is the single source of truth. It lists all the variables provided via Airflow's execution-time context — the things you can put inside {{ ... }} in any templated field.If you go there right now and Ctrl-F search for ds, you'll find the exact definitions I gave you. Same for ts_nodash, data_interval_start, logical_date, etc.

OR

Open your DAG in the UI → click on a task → click the "Rendered Template" tab. This shows you the actual value each templated field got for that specific run. It's the fastest way to verify "did ts_nodash give me what I expected?" without re-running anything.